# SPaRG-MF: Spiking Max-Former - CIFAR-100 Training

This notebook trains **SPaRG-MF** on **CIFAR-100**.

**Setup:** Go to `Runtime > Change runtime type > GPU` (T4 or better), then run all cells.

## 1. Get Code from GitHub

In [ ]:
import os

# Clone the repository if it doesn't exist
if not os.path.isdir("SPaRG-MF"):
    !git clone https://github.com/ParthKulkarn1/SPaRG-MF.git

PROJECT_DIR = "/content/SPaRG-MF"
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. (Optional) Mount Google Drive for Checkpoints

Colab deletes all files when the runtime disconnects. If you want your checkpoints to survive, mount Google Drive.

In [ ]:
OUTPUT_DIR = "/content/SPaRG-MF/output"

# Uncomment the following lines to save checkpoints to your Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# OUTPUT_DIR = "/content/drive/MyDrive/SPaRG-MF-Checkpoints"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {OUTPUT_DIR}")

## 3. Install Dependencies

In [ ]:
!pip install -q "timm>=0.9.0" "spikingjelly>=0.0.0.0.14" einops

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 4. Verify Model Builds

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from models.maxformer_snn import SpikingMaxFormer
from spikingjelly.activation_based import functional

test_model = SpikingMaxFormer(
    img_size=32, in_channels=3, num_classes=100,
    embed_dims=128, depths=[1, 1, 3], num_heads=4,
    time_steps=4, enable_head_gate=True,
    enable_token_gate=True, enable_mixed_prec=True,
    token_keep_ratio=[0.9, 0.7, 0.5], gate_type="magnitude",
    batch_averaged=False, time_dependent=True,
).cuda()

params = sum(p.numel() for p in test_model.parameters())
print(f"Model built! Parameters: {params / 1e6:.2f} M")

dummy = torch.randn(2, 3, 32, 32).cuda()
with torch.no_grad():
    out = test_model(dummy)
    functional.reset_net(test_model)
print(f"Forward pass OK! {dummy.shape} -> {out.shape}")

del test_model, dummy, out
torch.cuda.empty_cache()

## 5. Training Configuration

In [ ]:
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
TIME_STEPS = 4
EMBED_DIMS = 128
DEPTHS = [1, 1, 3]
NUM_HEADS = 4
NUM_WORKERS = 2

## 6. Load CIFAR-100 Dataset

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

train_set = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_train)
test_set = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_test)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f"CIFAR-100: {len(train_set)} train / {len(test_set)} test")

## 7. Build Model

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

import torch.nn as nn
from models.maxformer_snn import SpikingMaxFormer
from spikingjelly.activation_based import functional
from engine.regularization import DiversityLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SpikingMaxFormer(
    img_size=32, in_channels=3, num_classes=100,
    embed_dims=EMBED_DIMS, depths=DEPTHS, num_heads=NUM_HEADS,
    time_steps=TIME_STEPS, mlp_ratio=4.0,
    enable_head_gate=True, enable_token_gate=True, enable_mixed_prec=True,
    token_keep_ratio=[0.9, 0.7, 0.5], gate_type="magnitude",
    batch_averaged=False, time_dependent=True,
).to(device)

diversity_loss_fn = DiversityLoss(weight=0.01)

params = sum(p.numel() for p in model.parameters())
print(f"Model on {device} | Parameters: {params / 1e6:.2f} M")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 8. Checkpoint Manager
This cell checks for existing checkpoints. It will automatically load them and prepare the training loop to resume from where it left off.

In [ ]:
import os

start_epoch = 1
best_acc = 0.0
ckpt_path = os.path.join(OUTPUT_DIR, "cifar100_checkpoint.pth")
best_path = os.path.join(OUTPUT_DIR, "cifar100_best.pth")

print("--- Checkpoint Status ---")
if os.path.isfile(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    print(f"✅ Found checkpoint: {ckpt_path}")
    print(f"   Epoch: {ckpt['epoch']}")
    print(f"   Validation Acc: {ckpt.get('acc', 0.0):.2f}%")
    print(f"   Best Acc so far: {ckpt.get('best_acc', 0.0):.2f}%")
    
    # Load the states
    model.load_state_dict(ckpt["state_dict"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    
    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt.get("best_acc", 0.0)
    print(f"\n➡️ Ready to resume training from epoch {start_epoch}.")
    del ckpt
else:
    print(f"❌ No checkpoint found at: {ckpt_path}")
    print("➡️ Ready to start training from scratch (Epoch 1).")

if os.path.isfile(best_path):
    print(f"\n🏆 Best model file exists at: {best_path}")

## 9. Train

In [ ]:
import time

print(f"Training SPaRG-MF on CIFAR-100 | Epochs {start_epoch}-{EPOCHS}")
print("=" * 60)

total_start = time.time()
for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss_cls = criterion(outputs, targets)
        loss_div = diversity_loss_fn(model)
        loss = loss_cls + loss_div
        loss.backward()
        optimizer.step()
        functional.reset_net(model)

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    scheduler.step()
    train_acc = 100.0 * correct / total

    # Evaluate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            functional.reset_net(model)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    test_acc = 100.0 * correct / total

    is_best = test_acc > best_acc
    best_acc = max(best_acc, test_acc)
    marker = " ** BEST" if is_best else ""
    print(f"Epoch {epoch:3d}/{EPOCHS} | {time.time()-t0:.0f}s | "
          f"Train: {train_acc:.1f}% | Test: {test_acc:.1f}% | Best: {best_acc:.1f}%{marker}")

    # Save checkpoint
    state = {
        "epoch": epoch,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "acc": test_acc,
        "best_acc": best_acc,
    }
    torch.save(state, ckpt_path)
    if is_best:
        torch.save(state, best_path)

    if epoch % 10 == 0:
        stats = model.count_spikes()
        if stats:
            for name, val in list(stats.items())[:4]:
                short = name.split(".")[-1]
                print(f"  Spike {short}: rate={val['avg_spike_rate']:.3f}")

elapsed = (time.time() - total_start) / 3600
print(f"\nTraining Complete! Best: {best_acc:.2f}% | Time: {elapsed:.1f}h")

## 10. Evaluate Best Model

In [ ]:
best_path = os.path.join(OUTPUT_DIR, "cifar100_best.pth")
if not os.path.isfile(best_path):
    print(f"No best model found at {best_path}")
else:
    ckpt = torch.load(best_path, map_location="cpu")
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded best model from epoch {ckpt['epoch']} (saved acc: {ckpt['acc']:.2f}%)")
    model.eval()

    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            functional.reset_net(model)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    final_acc = 100.0 * correct / total
    print(f"Final Validation Accuracy: {final_acc:.2f}%")

    stats = model.count_spikes()
    if stats:
        print("\nHomeostatic Spike Rates:")
        for k, v in stats.items():
            print(f"  {k}: rate={v['avg_spike_rate']:.4f}, threshold={v['v_threshold']:.4f}")

## 11. Post-Training Calibration & Multi-Metric Inference Pruning

This section runs post-training calibration on the trained model using `SNNInferencePruningEngine` and computes metrics (QK-alignment Q-value, Head Redundancy, Attention Entropy, Bernoulli Spike Entropy) to generate highly optimized static hardware bit-masks. It then sweeps thresholds to check accuracy vs head sparsity.

In [ ]:
from engine.inference_pruning import SNNInferencePruningEngine

print("--- Initiating Calibration Engine ---")
pruning_engine = SNNInferencePruningEngine(model, num_heads=NUM_HEADS)

# Run calibration on a subset of the test loader
pruning_engine.calibrate(test_loader, device, num_batches=10)

# Print a beautiful summary of all diagnostic metrics across heads
pruning_engine.interceptor.summary(num_heads=NUM_HEADS)

# Let's evaluate model accuracy and head sparsity under different metric masks!
metrics = {
    "Q-Value (QK-Align >= 0.1)": pruning_engine.generate_static_masks(metric="q_value", threshold=0.1),
    "Head Redundancy (Corr <= 0.6)": pruning_engine.generate_static_masks(metric="redundancy", threshold=0.6),
    "Attention Entropy (Ent <= 2.0)": pruning_engine.generate_static_masks(metric="entropy", threshold=2.0),
    "Spike Bern Entropy (Ent >= 0.2)": pruning_engine.generate_static_masks(metric="spike_entropy", threshold=0.2),
}

print("\n--- Evaluating Gated Pareto Sparsity vs. Accuracy ---")
for name, masks in metrics.items():
    acc, sparsity = pruning_engine.evaluate_mask_efficiency(test_loader, device, masks)
    print(f" Gating Policy: {name:<35} | Accuracy: {acc:.2f}% | Head Sparsity: {sparsity:.2f}%")